# 🧪 Invoice Extraction Pipeline - Full Test Suite
## اختبار كل مكون على حدة

الأهداف:
1. اختبار معالجة الصور (OpenCV)
2. اختبار الاتصال بـ Ollama
3. اختبار الاستخراج (Extraction)
4. اختبار التحليل (Parsing) والتحقق (Validation)
5. اختبار التكامل الكامل (End-to-End)

In [ ]:
# تثبيت المكتبات المطلوبة (لو مش موجودة)
!pip install opencv-python numpy pandas openpyxl requests Pillow watchdog tqdm

In [ ]:
import sys
import os
import cv2
import numpy as np
import json
import requests
import base64
import logging
import time
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt

# إعداد التسجيل لعرض الأخطاء في النوت بوك
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# استيراد موديولات المشروع (تأكد إن الملفات موجودة في نفس المجلد)
from config import *
from image_utils import preprocess_image, image_to_base64
from ollama_client import OllamaClient
from extractor import extract_invoice_data, EXTRACTION_PROMPT
from parser import Invoice, Item
from validator import validate_invoice, load_existing_data
from excel_writer import append_invoice

print("✅ All modules imported successfully!")

---
## 1. اختبار معالجة الصور (Image Preprocessing)

In [ ]:
# حدد مسار صورة موجودة في مجلد invoices
test_image_path = list(INVOICES_DIR.glob("*.jpg")) + list(INVOICES_DIR.glob("*.png"))
if not test_image_path:
    raise FileNotFoundError("❌ لا توجد صور في مجلد invoices/! ضع صورة واحدة على الأقل.")
test_image_path = test_image_path[0]
print(f"📸 جاري اختبار الصورة: {test_image_path.name}")

# تحميل الصورة الأصلية
original = cv2.imread(str(test_image_path))
original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

# تطبيق المعالجة المسبقة
processed = preprocess_image(str(test_image_path))
processed_rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)

# عرض الصور قبل وبعد
fig, axes = plt.subplots(1, 2, figsize=(15, 10))
axes[0].imshow(original_rgb)
axes[0].set_title(f"Original: {original.shape[1]}x{original.shape[0]}")
axes[0].axis('off')

axes[1].imshow(processed_rgb)
axes[1].set_title(f"Processed: {processed.shape[1]}x{processed.shape[0]}")
axes[1].axis('off')
plt.show()

# تحويل إلى Base64
b64 = image_to_base64(processed)
print(f"✅ تمت المعالجة بنجاح!")
print(f"   - الأبعاد الجديدة: {processed.shape[1]}x{processed.shape[0]}")
print(f"   - حجم Base64: {len(b64)} حرف (حوالي {len(b64)*3//4/1024:.2f} KB)")

---
## 2. اختبار الاتصال بـ Ollama (بدون إرسال صورة كبيرة)

In [ ]:
def test_ollama_connection():
    """اختبار بسيط: إرسال طلب نصي فقط للتحقق من أن السيرفر شغال والموديل محمل"""
    url = OLLAMA_URL
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": "Hello, are you working? Respond with 'OK'."}
        ],
        "stream": False
    }
    try:
        print(f"⏳ جاري الاتصال بـ {url} باستخدام الموديل {MODEL_NAME}...")
        resp = requests.post(url, json=payload, timeout=30)
        if resp.status_code == 200:
            data = resp.json()
            content = data.get("message", {}).get("content", "")
            print(f"✅ الاتصال ناجح! رد الموديل: {content[:100]}")
            return True
        else:
            print(f"❌ فشل الاتصال: HTTP {resp.status_code}")
            print(f"   الرسالة: {resp.text}")
            return False
    except requests.exceptions.ConnectionError:
        print("❌ لا يمكن الاتصال بـ Ollama. تأكد من تشغيل `ollama serve`")
        return False
    except Exception as e:
        print(f"❌ خطأ غير متوقع: {e}")
        return False

test_ollama_connection()

---
## 3. اختبار العميل (OllamaClient) بإرسال الصورة بعد تصغيرها جداً

In [ ]:
# نستخدم صورة مصغرة جداً لتجنب خطأ 500 بسبب الحجم
def test_ollama_with_small_image():
    client = OllamaClient()
    
    # جلب الصورة وتصغيرها يدوياً لأقصى حد (600 بكسل)
    img = cv2.imread(str(test_image_path))
    h, w = img.shape[:2]
    scale = min(600 / w, 600 / h)
    if scale < 1:
        new_w = int(w * scale)
        new_h = int(h * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    b64_small = image_to_base64(img)
    print(f"📦 حجم الصورة المصغرة: {len(b64_small)} حرف")
    
    # برومبت بسيط
    prompt = "Extract the total amount from this receipt. Return only JSON with key 'total'."
    
    try:
        print("⏳ إرسال طلب إلى Ollama...")
        response = client.chat(b64_small, prompt)
        content = response.get("message", {}).get("content", "")
        print(f"✅ رد الموديل:\n{content[:500]}")
        return True
    except Exception as e:
        print(f"❌ فشل الطلب: {e}")
        return False

test_ollama_with_small_image()

---
## 4. اختبار دالة الاستخراج الكاملة (Extractor) مع Retry

In [ ]:
# استخدام نفس الصورة لكن مع تقليل الحجم في config مؤقتاً
def test_full_extraction():
    # نعدل دالة resize مؤقتاً لاستخدام حد أقل في الاختبار
    from image_utils import resize_if_needed
    original_resize = resize_if_needed
    
    def resize_small(img, max_w=1200, max_h=1200):
        return original_resize(img, max_w, max_h)
    
    # استبدال الدالة مؤقتاً
    import image_utils
    image_utils.resize_if_needed = resize_small
    
    client = OllamaClient()
    print(f"⏳ استخراج البيانات من {test_image_path.name}...")
    result = extract_invoice_data(str(test_image_path), client)
    
    # إعادة الدالة الأصلية
    image_utils.resize_if_needed = original_resize
    
    if result:
        print("✅ تم الاستخراج بنجاح!")
        print(json.dumps(result, indent=2, ensure_ascii=False)[:1000])
    else:
        print("❌ فشل الاستخراج بعد المحاولات.")
    return result

extracted_data = test_full_extraction()

---
## 5. اختبار التحليل (Parser) والتحقق (Validator)

In [ ]:
if extracted_data:
    try:
        # تحويل إلى كائن Invoice
        invoice = Invoice.from_dict(extracted_data)
        print("✅ Parser نجح:")
        print(f"   - رقم الفاتورة: {invoice.invoice_number}")
        print(f"   - الإجمالي: {invoice.grand_total}")
        print(f"   - عدد الأصناف: {len(invoice.items)}")
        
        # التحقق من الصحة
        # نحتاج لـ existing_invoices و existing_hashes (فارغين في الاختبار)
        existing_invoices = set()
        existing_hashes = set()
        is_valid, error_msg = validate_invoice(invoice, existing_invoices, existing_hashes)
        
        if is_valid:
            print("✅ Validator: الفاتورة صحيحة!")
        else:
            print(f"❌ Validator: {error_msg}")
            
    except Exception as e:
        print(f"❌ فشل Parser أو Validator: {e}")
else:
    print("⏭️ تخطي اختبار Parser/Validator لعدم وجود بيانات مستخرجة.")

---
## 6. اختبار الكتابة إلى Excel (Append)

In [ ]:
if extracted_data:
    try:
        invoice = Invoice.from_dict(extracted_data)
        # محاكاة كتابة ملف Excel
        success = append_invoice(invoice)
        if success:
            print(f"✅ تمت الكتابة إلى {OUTPUT_FILE}")
        else:
            print("❌ فشل الكتابة إلى Excel")
    except Exception as e:
        print(f"❌ خطأ أثناء الكتابة: {e}")
else:
    print("⏭️ تخطي اختبار Excel لعدم وجود بيانات.")

---
## 7. اختبار شامل (Pipeline كامل) على كل الصور في المجلد

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def process_one_image(img_path):
    client = OllamaClient()
    data = extract_invoice_data(str(img_path), client)
    if data is None:
        return img_path.name, None, "Extraction failed"
    try:
        inv = Invoice.from_dict(data)
        existing_invoices, existing_hashes = set(), set()
        valid, msg = validate_invoice(inv, existing_invoices, existing_hashes)
        if not valid:
            return img_path.name, None, f"Validation failed: {msg}"
        return img_path.name, inv, "Success"
    except Exception as e:
        return img_path.name, None, f"Error: {e}"

all_images = list(INVOICES_DIR.glob("*"))  # أي امتداد
image_files = [f for f in all_images if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']]

if image_files:
    print(f"🔄 جاري اختبار {len(image_files)} صورة...")
    results = []
    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = {executor.submit(process_one_image, img): img for img in image_files}
        for future in tqdm(as_completed(futures), total=len(futures)):
            name, inv, status = future.result()
            results.append((name, status))
            print(f"  - {name}: {status}")
else:
    print("⚠️ لا توجد صور في مجلد invoices/.")

---
## 🔍 تشخيص خطأ 500 تحديداً

لو لسه في خطأ 500، الجاي هيجرب يبعت الصورة بـ **جودة أقل** و **حجم أصغر** ويطبع الـ Error Response من السيرفر

In [ ]:
def diagnose_500_error():
    img = cv2.imread(str(test_image_path))
    # تصغير قوي جداً (800 بكسل)
    h, w = img.shape[:2]
    scale = min(800 / w, 800 / h, 1.0)
    if scale < 1:
        img = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_AREA)
    
    # ضغط الصورة بجودة أقل (JPEG) عشان نقلل الحجم أكتر
    _, buffer = cv2.imencode('.jpg', img, [int(cv2.IMWRITE_JPEG_QUALITY), 70])
    b64_small = base64.b64encode(buffer).decode('utf-8')
    
    print(f"📦 حجم Base64 بعد الضغط: {len(b64_small):,} حرف")
    
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": EXTRACTION_PROMPT,
                "images": [b64_small]
            }
        ],
        "stream": False
    }
    
    try:
        resp = requests.post(OLLAMA_URL, json=payload, timeout=120)
        print(f"🔴 Status Code: {resp.status_code}")
        if resp.status_code != 200:
            print(f"📄 Response Body:\n{resp.text}")
        else:
            print(f"✅ نجح! الرد:\n{resp.json().get('message', {}).get('content', '')[:500]}")
    except Exception as e:
        print(f"❌ استثناء: {e}")

diagnose_500_error()

---
## ✅ الخلاصة

- لو **اختبار 1** (معالجة الصور) نجح → المشكلة مش في OpenCV.
- لو **اختبار 2** (النصي) نجح و **اختبار 3** (الصورة المصغرة) فشل → المشكلة في حجم الصورة أو الموديل نفسه.
- لو **اختبار التشخيص الأخير** طبع رسالة خطأ من السيرفر (زي `model not found` أو `input too large`) → هنعرف نصلحها.

💡 **نصيحة**: غير `MAX_WIDTH` و `MAX_HEIGHT` في `config.py` لـ `1200` بدل `2000`، وقلل `RETRY_ATTEMPTS` لـ `1` عشان تسريع الاختبار.